**Assignment: End-to-End ML Pipeline on Tesla Deliveries & Pricing Data**

Build a complete Machine Learning pipeline that:

Preprocesses the data
Performs EDA
Creates new features
Trains regression models
Performs hyperparameter tuning
Forecasts future deliveries using time series analysis

**Import Libraries**

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from statsmodels.tsa.holtwinters import ExponentialSmoothing

**Load Dataset**

In [ ]:
df = pd.read_csv("/content/tesla_deliveries_dataset_2015_2025.csv")

print(df.head())
print(df.info())
print(df.describe())

Data Preprocessing

Missing Values

In [ ]:
print(df.isnull().sum())

In [ ]:
df.fillna(df.median(numeric_only=True), inplace=True)

In [ ]:
print("Duplicates:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

**Exploratory Data Analysis (EDA)**

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["Estimated_Deliveries"], kde=True)
plt.title("Distribution of Deliveries")
plt.show()

**Average Price by Model**

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(
    data=df,
    x="Model",
    y="Avg_Price_USD"
)
plt.xticks(rotation=45)
plt.show()

**Deliveries by Region**

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    data=df,
    x="Region",
    y="Estimated_Deliveries"
)

plt.xticks(rotation=45)
plt.show()

**Correlation Matrix**

In [ ]:
plt.figure(figsize=(10,6))

sns.heatmap(
    df.select_dtypes(include=np.number).corr(),
    annot=True,
    cmap="coolwarm"
)

plt.show()

**Feature Engineering**

Create Date Column

In [ ]:
df["Date"] = pd.to_datetime(
    df["Year"].astype(str)
    + "-"
    + df["Month"].astype(str)
)

Extract Features

In [ ]:
df["Quarter"] = df["Date"].dt.quarter

df["Year_Month"] = (
    df["Year"]*100
    + df["Month"]
)

Production Efficiency

In [ ]:
df["Production_Efficiency"] = (
    df["Estimated_Deliveries"]
    / df["Production_Units"]
)

Price per Kilometer

In [ ]:
df["Price_per_km"] = (
    df["Avg_Price_USD"]
    / df["Range_km"]
)

**Regression Problem**

**Predict Deliveries**

Target:

In [ ]:
y = df["Estimated_Deliveries"]

Features:

In [ ]:
X = df.drop(
    columns=[
        "Estimated_Deliveries",
        "Date"
    ]
)

**Separate Columns**

In [ ]:
cat_cols = X.select_dtypes(
    include="object"
).columns

num_cols = X.select_dtypes(
    exclude="object"
).columns

**Build ML Pipeline**

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

**Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

**Linear Regression**

In [ ]:
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

lr_pipeline.fit(X_train, y_train)

preds = lr_pipeline.predict(X_test)

**Evaluation**

In [ ]:
print("MAE:",
      mean_absolute_error(y_test,preds))

print("RMSE:",
      np.sqrt(mean_squared_error(y_test,preds)))

print("R2:",
      r2_score(y_test,preds))

**Random Forest Regression**

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_preds = rf_pipeline.predict(X_test)

**Evaluate**

In [ ]:
print("R2:",
      r2_score(y_test, rf_preds))

**Hyperparameter Tuning**

In [ ]:
param_grid = {
    "model__n_estimators":[100,200],
    "model__max_depth":[5,10,15],
    "model__min_samples_split":[2,5]
}

In [ ]:
grid = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train,y_train)

print(grid.best_params_)

**Best Model**

In [ ]:
best_model = grid.best_estimator_

best_preds = best_model.predict(X_test)

print(
    "Best R2:",
    r2_score(y_test,best_preds)
)

**Time Series Forecasting**

Monthly Deliveries

In [ ]:
ts = (
    df.groupby("Date")
    ["Estimated_Deliveries"]
    .sum()
    .sort_index()
)

**Plot**

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(ts)

plt.title(
    "Tesla Deliveries Over Time"
)

plt.show()


**Forecast**

In [ ]:
model = ExponentialSmoothing(
    ts,
    trend="add",
    seasonal="add",
    seasonal_periods=12
)

fit = model.fit()

**Next 12 Months Forecast**

In [ ]:
forecast = fit.forecast(12)

print(forecast)

**Visualization**

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    ts,
    label="Actual"
)

plt.plot(
    forecast,
    label="Forecast"
)

plt.legend()
plt.show()

**Designed and implemented an end-to-end Machine Learning pipeline on Tesla deliveries and pricing data. The project included data preprocessing, exploratory data analysis, feature engineering, regression modeling, hyperparameter tuning using GridSearchCV, and time-series forecasting using Exponential Smoothing. The final model successfully predicted vehicle deliveries and provided future delivery forecasts, enabling data-driven business insights.**